In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Update this path if needed
DATASET_PATH = '/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv'

df = pd.read_csv(DATASET_PATH)
print('Shape:', df.shape)
print(df.head(3))

target_col = 'Diabetes_binary'
y = df[target_col].astype(int).values
X_df = df.drop(columns=[target_col])

# Safety for any unexpected non-numeric fields
object_cols = X_df.select_dtypes(include=['object']).columns
if len(object_cols) > 0:
    X_df = pd.get_dummies(X_df, columns=object_cols, drop_first=True)

X_df = X_df.apply(pd.to_numeric, errors='coerce')
X_df = X_df.fillna(X_df.median(numeric_only=True))
X = X_df.values.astype(np.float32)

print('Features:', X.shape[1])
print('Samples:', X.shape[0])
print(f'Class balance -> No Diabetes: {(y==0).sum()}, Diabetes: {(y==1).sum()}')
print(f'Positive rate: {(y==1).mean():.4f}')

# 64/16/20 split (train/val/test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, random_state=SEED, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')

In [ ]:
class DiabetesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = DiabetesDataset(X_train, y_train)
val_dataset = DiabetesDataset(X_val, y_val)
test_dataset = DiabetesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)


class DiabetesNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x)


model = DiabetesNet(input_dim=X_train.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

print(model)

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            probs = torch.sigmoid(logits)

            total_loss += criterion(logits, y_b).item()
            all_probs.extend(probs.cpu().numpy().ravel())
            all_labels.extend(y_b.cpu().numpy().ravel())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels).astype(int)
    roc_auc = roc_auc_score(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)

    return total_loss / len(loader), all_labels, all_probs, roc_auc, pr_auc


best_val_loss = float('inf')
patience = 10
patience_counter = 0
epochs = 100

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_labels, val_probs, val_roc_auc, val_pr_auc = eval_epoch(
        model, val_loader, criterion
    )

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_alex5050_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch + 1}')
            break

    if (epoch + 1) % 5 == 0:
        print(
            f'Epoch {epoch + 1:3d} | Train Loss: {train_loss:.4f} | '
            f'Val Loss: {val_loss:.4f} | Val ROC-AUC: {val_roc_auc:.4f} | Val PR-AUC: {val_pr_auc:.4f}'
        )

In [ ]:
def best_threshold_by_f1(labels, probs):
    thresholds = np.linspace(0.05, 0.95, 91)
    best_thr, best_f1 = 0.5, -1.0

    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return float(best_thr), float(best_f1)


model.load_state_dict(torch.load('best_alex5050_model.pth', map_location=device))
model.eval()

_, val_labels, val_probs, val_roc_auc, val_pr_auc = eval_epoch(model, val_loader, criterion)
best_thr, best_val_f1 = best_threshold_by_f1(val_labels, val_probs)

_, test_labels, test_probs, test_roc_auc, test_pr_auc = eval_epoch(model, test_loader, criterion)
test_preds = (test_probs >= best_thr).astype(int)

test_precision = precision_score(test_labels, test_preds, zero_division=0)
test_recall = recall_score(test_labels, test_preds, zero_division=0)
test_f1 = f1_score(test_labels, test_preds, zero_division=0)

print(f'Validation ROC-AUC: {val_roc_auc:.4f}')
print(f'Validation PR-AUC : {val_pr_auc:.4f}')
print(f'Chosen threshold (best val F1): {best_thr:.2f} | Val F1: {best_val_f1:.4f}\n')

print(f'Test ROC-AUC: {test_roc_auc:.4f}')
print(f'Test PR-AUC : {test_pr_auc:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall   : {test_recall:.4f}')
print(f'Test F1       : {test_f1:.4f}\n')
print(classification_report(test_labels, test_preds, target_names=['No Diabetes', 'Diabetes'], zero_division=0))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No Diabetes', 'Diabetes'],
    yticklabels=['No Diabetes', 'Diabetes'],
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix @ threshold={best_thr:.2f}')
plt.tight_layout()
plt.show()

precisions, recalls, _ = precision_recall_curve(test_labels, test_probs)
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions, linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Test Precision-Recall Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

import joblib
joblib.dump(scaler, 'alex5050_scaler.joblib')
print('Saved model: best_alex5050_model.pth')
print('Saved scaler: alex5050_scaler.joblib')

In [ ]:
# Package model artifacts for easy download
import os
import zipfile
from pathlib import Path

try:
    from IPython.display import FileLink, display
except Exception:
    FileLink = None
    display = print

model_file = Path('best_alex5050_model.pth')
scaler_file = Path('alex5050_scaler.joblib')
bundle_file = Path('alex5050_model_bundle.zip')

missing = [str(p) for p in [model_file, scaler_file] if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Run the training/evaluation cell first to generate them."
    )

with zipfile.ZipFile(bundle_file, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(model_file, arcname=model_file.name)
    zf.write(scaler_file, arcname=scaler_file.name)

print(f"Created: {bundle_file.resolve()}")
print("Contents:", [model_file.name, scaler_file.name])

# Kaggle convenience copy
kaggle_working = Path('/kaggle/working')
if kaggle_working.exists():
    kaggle_bundle = kaggle_working / bundle_file.name
    kaggle_bundle.write_bytes(bundle_file.read_bytes())
    print(f"Copied to: {kaggle_bundle}")

if FileLink is not None:
    display(FileLink(str(bundle_file)))